# 05 — Gold star schema

Customer-facing star schema modeled from the conformed silver tables. Each table is loaded incrementally with `MERGE` (upsert on its surrogate key), the same pattern as silver, so a re-run refreshes changed rows and inserts new ones instead of rewriting the table. Keys are declared as RELY PRIMARY KEY / FOREIGN KEY constraints so AI/BI dashboards and Genie can infer the joins automatically.

**Dimensions**
- **dim_team**: from `silver_teams`, keyed `team_key`, carrying `mlbam_team_id` (the MLBAM cross-source key).
- **dim_player**: from `silver_players`, keyed `player_key`, carrying `mlbam_player_id`.
- **dim_date**: conformed calendar (keyed `date_key` as `yyyyMMdd`) covering every date the facts reference.
- **dim_venue**: ballparks, from `silver_venues`, keyed `venue_key`.
- **dim_competition**: from `silver_competitions`, keyed `competition_key`.
- **dim_umpire**: from `silver_umpires`, keyed `umpire_key`, carrying `mlbam_umpire_id`.
- **dim_session**: practice sessions (parent of practice events), from `silver_practice_sessions`, keyed `session_key`.

**Game facts**
- **fact_game**: one row per game. Final score, total runs, run differential, and winner are derived by aggregating the running score on `silver_events` (score is monotonic, so `MAX` per game is the final). FKs to home/away/winner `dim_team`, `dim_venue`, `dim_competition`, `dim_date`.
- **fact_event**: one row per game event. FKs to `fact_game`, batter `dim_player`, `dim_team`, `dim_umpire`, `dim_date`.
- **fact_pitch**: one row per pitch (`silver_events_pitch_subset`); same FK spine as `fact_event`.

**Practice facts**
- **fact_practice_event**: one row per practice event (wide pass-through of all `silver_practice_events` columns); FKs to `dim_session`, batter `dim_player`, `dim_team`, `dim_date`.
- **fact_practice_workout**: one row per training-workout group (`silver_practice_training_workout`); FKs to `dim_session`, `dim_team`, `dim_date`.

**Roster fact**
- **fact_roster_stint**: player-team stints over time (`silver_players_teamhistory`); FKs to `dim_team` and `dim_date` (start/end). No `dim_player` FK, since the payload has no player id (join by name if needed).

Surrogate keys are `md5()` of the Synergy id, so a fact's `md5(<entity>_id)` lines up with the matching dimension's `md5(id)` key without a lookup join. Cells run top-to-bottom: dimensions before the facts that reference them.

> **Known source gaps (not modeled here):** the game/practice event payloads carry no pitcher id (only `pitcher_info_*` attributes), so facts link the batter only; and `players_teamhistory` has no player id. `fact_game` scores reflect ingested events: they are null for games whose events weren't pulled, and a partial event window can make a game look tied. The `events_defense_subset` / `events_game_state_subset` projections and the `search_*` endpoints are not modeled (redundant grain or operational). The `leagues`, `divisions`, and `conferences` reference entities stay denormalized as text on `dim_team`. Column and table `COMMENT`s for Genie are added in notebook 06.

In [ ]:
# Job parameter bridge: when run as a Databricks Job task, the bundle's base_parameters arrive as
# notebook widgets. Mirror them into os.environ before the setup cell reads them, so the same
# os.getenv(...) logic works as a job, in a workspace, or locally via .env. dbutils is absent under
# local Databricks Connect, so the except clause handles that case.
import os
for _k in ("UC_CATALOG", "UC_SCHEMA"):
    try:
        _v = dbutils.widgets.get(_k)            # noqa: F821 — dbutils is injected in Databricks
        if _v:
            os.environ[_k] = _v
    except Exception:
        pass

In [ ]:
# Dual-mode setup: runs locally via Databricks Connect and inside a Databricks Git Folder.
import os, json
try:
    from dotenv import load_dotenv; load_dotenv()
except Exception:
    pass
try:
    spark  # already present inside a Databricks workspace notebook
except NameError:
    from databricks.connect import DatabricksSession
    spark = DatabricksSession.builder.serverless().getOrCreate()

UC_CATALOG    = os.getenv("UC_CATALOG", "main")
UC_SCHEMA     = os.getenv("UC_SCHEMA", "synergy_baseball_demo")
BRONZE_SCHEMA = f"{UC_SCHEMA}_bronze"
SILVER_SCHEMA = f"{UC_SCHEMA}_silver"
GOLD_SCHEMA   = f"{UC_SCHEMA}_gold"
VOLUME_NAME   = "raw_data"
VOLUME_PATH   = f"/Volumes/{UC_CATALOG}/{BRONZE_SCHEMA}/{VOLUME_NAME}"
B, S, G = f"{UC_CATALOG}.{BRONZE_SCHEMA}", f"{UC_CATALOG}.{SILVER_SCHEMA}", f"{UC_CATALOG}.{GOLD_SCHEMA}"
print("target:", B, "|", S, "|", G)

In [ ]:
# Incremental gold via MERGE. Each table is upserted on its surrogate key; a re-run refreshes changed
# rows and inserts new ones. WITH SCHEMA EVOLUTION absorbs added columns (the autoMerge config is
# unavailable on serverless, so evolution is requested per statement).

def upsert(table: str, select_sql: str, keys: list):
    """Create `table` from the select if absent, then MERGE on `keys` (idempotent upsert)."""
    spark.sql(f"CREATE TABLE IF NOT EXISTS {table} AS SELECT * FROM ({select_sql}) WHERE 1=0")
    on = " AND ".join(f"t.{k} <=> s.{k}" for k in keys)
    spark.sql(f"MERGE WITH SCHEMA EVOLUTION INTO {table} t USING ({select_sql}) s ON {on} "
              "WHEN MATCHED THEN UPDATE SET * WHEN NOT MATCHED THEN INSERT *")

def constrain(*statements: str):
    """Apply NOT NULL / PK / FK statements once; ignore 'already exists' so re-runs are idempotent
    (the tables persist across runs now, unlike CREATE OR REPLACE)."""
    for sql in statements:
        try:
            spark.sql(sql)
        except Exception as ex:
            if "already exists" not in str(ex).lower() and "already present" not in str(ex).lower():
                raise


In [ ]:
# dim_team (carries external_id_mlbam = the MLBAM cross-source join key)
upsert(f"{G}.dim_team", f"""
    SELECT
        md5(id)              AS team_key,
        id                   AS synergy_team_id,
        external_id_mlbam    AS mlbam_team_id,   -- MLBAM id: the cross-source conformance key
        name, name_abbrev, league_name, division_name, conference_name
    FROM {S}.silver_teams
""", ["team_key"])
print("dim_team:", spark.table(f"{G}.dim_team").count())

# RELY PK/FK pattern: declare keys so AI/BI dashboards and Genie can infer joins automatically.
# NOT NULL is required before a column can be a PRIMARY KEY.
constrain(
    f"ALTER TABLE {G}.dim_team ALTER COLUMN team_key SET NOT NULL",
    f"ALTER TABLE {G}.dim_team ADD CONSTRAINT pk_dim_team PRIMARY KEY (team_key) RELY",
)

In [ ]:
# dim_player (carries external_id_mlbam = the MLBAM cross-source join key)
upsert(f"{G}.dim_player", f"""
    SELECT
        md5(id)               AS player_key,
        id                    AS synergy_player_id,
        external_id_mlbam     AS mlbam_player_id,   -- MLBAM id: the cross-source conformance key
        concat_ws(' ', name_first, name_last) AS player_name,
        name_first, name_last,
        birth_date, country,
        bats_handedness, throws_handedness,
        position, most_recent_position,
        most_recent_team_id, most_recent_team_name,
        league_name
    FROM {S}.silver_players
""", ["player_key"])
print("dim_player:", spark.table(f"{G}.dim_player").count())

constrain(
    f"ALTER TABLE {G}.dim_player ALTER COLUMN player_key SET NOT NULL",
    f"ALTER TABLE {G}.dim_player ADD CONSTRAINT pk_dim_player PRIMARY KEY (player_key) RELY",
)

In [ ]:
# dim_date: conformed calendar built from every date the fact tables reference (games, events,
# practice sessions/events, and roster stints) so no fact's date_key points at a missing dimension row.
upsert(f"{G}.dim_date", f"""
    WITH all_dates AS (
        SELECT coalesce(try_cast(left(game_date, 10) AS date), try_to_date(game_date, 'M/d/yyyy'))      AS d FROM {S}.silver_games              WHERE game_date         IS NOT NULL
        UNION SELECT coalesce(try_cast(left(game_game_date, 10) AS date), try_to_date(game_game_date, 'M/d/yyyy'))  FROM {S}.silver_events             WHERE game_game_date    IS NOT NULL
        UNION SELECT coalesce(try_cast(left(practice_date, 10) AS date), try_to_date(practice_date, 'M/d/yyyy'))   FROM {S}.silver_practice_sessions  WHERE practice_date     IS NOT NULL
        UNION SELECT coalesce(try_cast(left(session_game_date, 10) AS date), try_to_date(session_game_date, 'M/d/yyyy'))FROM {S}.silver_practice_events    WHERE session_game_date IS NOT NULL
        UNION SELECT coalesce(try_cast(left(start_date, 10) AS date), try_to_date(start_date, 'M/d/yyyy'))      FROM {S}.silver_players_teamhistory WHERE start_date       IS NOT NULL
        UNION SELECT coalesce(try_cast(left(end_date, 10) AS date), try_to_date(end_date, 'M/d/yyyy'))        FROM {S}.silver_players_teamhistory WHERE end_date         IS NOT NULL
    )
    SELECT
        cast(date_format(d, 'yyyyMMdd') AS int) AS date_key,
        d                                       AS full_date,
        year(d)                                 AS year,
        quarter(d)                              AS quarter,
        month(d)                                AS month,
        date_format(d, 'MMMM')                  AS month_name,
        day(d)                                  AS day_of_month,
        weekofyear(d)                           AS week_of_year,
        date_format(d, 'EEEE')                  AS day_name,
        dayofweek(d) IN (1, 7)                  AS is_weekend
    FROM (SELECT DISTINCT d FROM all_dates WHERE d IS NOT NULL)
""", ["date_key"])
print("dim_date:", spark.table(f"{G}.dim_date").count())

constrain(
    f"ALTER TABLE {G}.dim_date ALTER COLUMN date_key SET NOT NULL",
    f"ALTER TABLE {G}.dim_date ADD CONSTRAINT pk_dim_date PRIMARY KEY (date_key) RELY",
)

In [ ]:
# dim_venue (ballparks)
upsert(f"{G}.dim_venue", f"""
    SELECT
        md5(id)       AS venue_key,
        id            AS synergy_venue_id,
        name, name_abbrev, city
    FROM {S}.silver_venues
""", ["venue_key"])
print("dim_venue:", spark.table(f"{G}.dim_venue").count())

constrain(
    f"ALTER TABLE {G}.dim_venue ALTER COLUMN venue_key SET NOT NULL",
    f"ALTER TABLE {G}.dim_venue ADD CONSTRAINT pk_dim_venue PRIMARY KEY (venue_key) RELY",
)

In [ ]:
# dim_session — practice sessions (the parent of every practice event); FKs to dim_team + dim_date
upsert(f"{G}.dim_session", f"""
    SELECT
        md5(id)                AS session_key,
        id                     AS synergy_session_id,
        md5(home_team_id)      AS home_team_key,
        md5(away_team_id)      AS away_team_key,
        cast(date_format(coalesce(try_cast(left(practice_date, 10) AS date), try_to_date(practice_date, 'M/d/yyyy')), 'yyyyMMdd') AS int) AS date_key,
        home_team_name, away_team_name,
        practice_date, practice_time, season, double_header,
        practice_session_type,
        league_name
    FROM {S}.silver_practice_sessions
""", ["session_key"])
print("dim_session:", spark.table(f"{G}.dim_session").count())

constrain(
    f"ALTER TABLE {G}.dim_session ALTER COLUMN session_key SET NOT NULL",
    f"ALTER TABLE {G}.dim_session ADD CONSTRAINT pk_dim_session PRIMARY KEY (session_key) RELY",
    f"ALTER TABLE {G}.dim_session ADD CONSTRAINT fk_session_home_team FOREIGN KEY (home_team_key) REFERENCES {G}.dim_team(team_key) RELY",
    f"ALTER TABLE {G}.dim_session ADD CONSTRAINT fk_session_away_team FOREIGN KEY (away_team_key) REFERENCES {G}.dim_team(team_key) RELY",
    f"ALTER TABLE {G}.dim_session ADD CONSTRAINT fk_session_date      FOREIGN KEY (date_key)      REFERENCES {G}.dim_date(date_key) RELY",
)

In [ ]:
# dim_competition — from silver_competitions
upsert(f"{G}.dim_competition", f"""
    SELECT
        md5(id)        AS competition_key,
        id             AS synergy_competition_id,
        name           AS competition_name,
        league_name
    FROM {S}.silver_competitions
""", ["competition_key"])
print("dim_competition:", spark.table(f"{G}.dim_competition").count())

constrain(
    f"ALTER TABLE {G}.dim_competition ALTER COLUMN competition_key SET NOT NULL",
    f"ALTER TABLE {G}.dim_competition ADD CONSTRAINT pk_dim_competition PRIMARY KEY (competition_key) RELY",
)

In [ ]:
# dim_umpire — from silver_umpires (carries mlbam_umpire_id = the MLBAM cross-source key)
upsert(f"{G}.dim_umpire", f"""
    SELECT
        md5(id)            AS umpire_key,
        id                 AS synergy_umpire_id,
        external_id_mlbam  AS mlbam_umpire_id,
        concat_ws(' ', name_first, name_last) AS umpire_name,
        name_first, name_last
    FROM {S}.silver_umpires
""", ["umpire_key"])
print("dim_umpire:", spark.table(f"{G}.dim_umpire").count())

constrain(
    f"ALTER TABLE {G}.dim_umpire ALTER COLUMN umpire_key SET NOT NULL",
    f"ALTER TABLE {G}.dim_umpire ADD CONSTRAINT pk_dim_umpire PRIMARY KEY (umpire_key) RELY",
)

In [ ]:
# fact_game — one row per game; FKs to home/away/winner dim_team, dim_venue, dim_competition, dim_date.
# Scores aren't in silver_games; they live as the running score on each silver_events row. A baseball
# score only ever increases, so the final score is MAX(running score) per game. This tolerates missing
# last-event flags and unordered rows. Sourced from silver_events directly to avoid a fact_event dependency.
upsert(f"{G}.fact_game", f"""
    WITH game_score AS (
        SELECT game_id,
               max(home_team_score) AS home_score,
               max(away_team_score) AS away_score
        FROM   {S}.silver_events
        GROUP  BY game_id
    )
    SELECT
        md5(g.id)              AS game_key,
        g.id                   AS synergy_game_id,
        g.external_id_mlbam    AS mlbam_game_id,
        md5(g.home_team_id)    AS home_team_key,
        md5(g.away_team_id)    AS away_team_key,
        md5(g.venue_id)        AS venue_key,
        md5(g.competition_id)  AS competition_key,
        cast(date_format(coalesce(try_cast(left(g.game_date, 10) AS date), try_to_date(g.game_date, 'M/d/yyyy')), 'yyyyMMdd') AS int) AS date_key,
        g.season, g.game_number, g.double_header,
        g.game_level, g.game_type, g.game_time_zone,
        g.competition_name,
        g.home_team_name, g.away_team_name,
        -- derived score measures (null when no events were ingested for the game)
        gs.home_score,
        gs.away_score,
        (gs.home_score + gs.away_score) AS total_runs,
        (gs.home_score - gs.away_score) AS run_differential,
        CASE WHEN gs.home_score > gs.away_score THEN md5(g.home_team_id)
             WHEN gs.away_score > gs.home_score THEN md5(g.away_team_id)
        END                             AS winner_team_key,
        (gs.home_score > gs.away_score) AS is_home_win
    FROM {S}.silver_games g
    LEFT JOIN game_score gs ON g.id = gs.game_id
""", ["game_key"])
print("fact_game:", spark.table(f"{G}.fact_game").count())

constrain(
    f"ALTER TABLE {G}.fact_game ALTER COLUMN game_key SET NOT NULL",
    f"ALTER TABLE {G}.fact_game ADD CONSTRAINT pk_fact_game PRIMARY KEY (game_key) RELY",
    f"ALTER TABLE {G}.fact_game ADD CONSTRAINT fk_game_home_team   FOREIGN KEY (home_team_key)   REFERENCES {G}.dim_team(team_key)               RELY",
    f"ALTER TABLE {G}.fact_game ADD CONSTRAINT fk_game_away_team   FOREIGN KEY (away_team_key)   REFERENCES {G}.dim_team(team_key)               RELY",
    f"ALTER TABLE {G}.fact_game ADD CONSTRAINT fk_game_winner      FOREIGN KEY (winner_team_key) REFERENCES {G}.dim_team(team_key)               RELY",
    f"ALTER TABLE {G}.fact_game ADD CONSTRAINT fk_game_venue       FOREIGN KEY (venue_key)       REFERENCES {G}.dim_venue(venue_key)             RELY",
    f"ALTER TABLE {G}.fact_game ADD CONSTRAINT fk_game_competition FOREIGN KEY (competition_key) REFERENCES {G}.dim_competition(competition_key) RELY",
    f"ALTER TABLE {G}.fact_game ADD CONSTRAINT fk_game_date        FOREIGN KEY (date_key)        REFERENCES {G}.dim_date(date_key)               RELY",
)

In [ ]:
# fact_event — one row per game event; FKs to fact_game, batter dim_player, dim_team, dim_umpire, dim_date
upsert(f"{G}.fact_event", f"""
    SELECT
        md5(id)                   AS event_key,
        id                        AS synergy_event_id,
        md5(game_id)              AS game_key,
        md5(batter_id)            AS batter_key,
        md5(game_home_team_id)    AS home_team_key,
        md5(game_away_team_id)    AS away_team_key,
        md5(umpire_home_plate_id) AS umpire_key,
        cast(date_format(coalesce(try_cast(left(game_game_date, 10) AS date), try_to_date(game_game_date, 'M/d/yyyy')), 'yyyyMMdd') AS int) AS date_key,
        inning, inning_top, outs,
        count_balls, count_strikes,
        event_type, plate_appearance_result,
        pitch_pitch_speed_mph, pitch_pitch_kind, pitch_pitch_result,
        contact_exit_speed_mph, contact_launch_angle_degrees, contact_distance_feet,
        home_team_score, away_team_score,
        last_event_in_atbat
    FROM {S}.silver_events
""", ["event_key"])
print("fact_event:", spark.table(f"{G}.fact_event").count())

constrain(
    f"ALTER TABLE {G}.fact_event ALTER COLUMN event_key SET NOT NULL",
    f"ALTER TABLE {G}.fact_event ADD CONSTRAINT pk_fact_event PRIMARY KEY (event_key) RELY",
    f"ALTER TABLE {G}.fact_event ADD CONSTRAINT fk_event_game      FOREIGN KEY (game_key)      REFERENCES {G}.fact_game(game_key)    RELY",
    f"ALTER TABLE {G}.fact_event ADD CONSTRAINT fk_event_batter    FOREIGN KEY (batter_key)    REFERENCES {G}.dim_player(player_key) RELY",
    f"ALTER TABLE {G}.fact_event ADD CONSTRAINT fk_event_home_team FOREIGN KEY (home_team_key) REFERENCES {G}.dim_team(team_key)     RELY",
    f"ALTER TABLE {G}.fact_event ADD CONSTRAINT fk_event_away_team FOREIGN KEY (away_team_key) REFERENCES {G}.dim_team(team_key)     RELY",
    f"ALTER TABLE {G}.fact_event ADD CONSTRAINT fk_event_umpire    FOREIGN KEY (umpire_key)    REFERENCES {G}.dim_umpire(umpire_key) RELY",
    f"ALTER TABLE {G}.fact_event ADD CONSTRAINT fk_event_date      FOREIGN KEY (date_key)      REFERENCES {G}.dim_date(date_key)     RELY",
)

In [ ]:
# fact_pitch — one row per pitch (silver_events_pitch_subset); same FK spine as fact_event
upsert(f"{G}.fact_pitch", f"""
    SELECT
        md5(id)                AS pitch_key,
        id                     AS synergy_event_id,
        md5(game_id)           AS game_key,
        md5(batter_id)         AS batter_key,
        md5(game_home_team_id) AS home_team_key,
        md5(game_away_team_id) AS away_team_key,
        cast(date_format(coalesce(try_cast(left(game_game_date, 10) AS date), try_to_date(game_game_date, 'M/d/yyyy')), 'yyyyMMdd') AS int) AS date_key,
        inning, inning_top, outs,
        count_balls, count_strikes,
        pitch_num_in_plate_appearance,
        pitch_pitch_speed_mph, pitch_px, pitch_pz, pitch_pitch_zone,
        pitch_pitch_kind, pitch_pitch_result, pitch_pitch_result_modifier,
        batter_info_batting_side, pitcher_info_pitching_side,
        event_type, plate_appearance_result,
        contact_contact_type, contact_field_zone,
        last_event_in_atbat
    FROM {S}.silver_events_pitch_subset
""", ["pitch_key"])
print("fact_pitch:", spark.table(f"{G}.fact_pitch").count())

constrain(
    f"ALTER TABLE {G}.fact_pitch ALTER COLUMN pitch_key SET NOT NULL",
    f"ALTER TABLE {G}.fact_pitch ADD CONSTRAINT pk_fact_pitch PRIMARY KEY (pitch_key) RELY",
    f"ALTER TABLE {G}.fact_pitch ADD CONSTRAINT fk_pitch_game      FOREIGN KEY (game_key)      REFERENCES {G}.fact_game(game_key)   RELY",
    f"ALTER TABLE {G}.fact_pitch ADD CONSTRAINT fk_pitch_batter    FOREIGN KEY (batter_key)    REFERENCES {G}.dim_player(player_key) RELY",
    f"ALTER TABLE {G}.fact_pitch ADD CONSTRAINT fk_pitch_home_team FOREIGN KEY (home_team_key) REFERENCES {G}.dim_team(team_key)    RELY",
    f"ALTER TABLE {G}.fact_pitch ADD CONSTRAINT fk_pitch_away_team FOREIGN KEY (away_team_key) REFERENCES {G}.dim_team(team_key)    RELY",
    f"ALTER TABLE {G}.fact_pitch ADD CONSTRAINT fk_pitch_date      FOREIGN KEY (date_key)      REFERENCES {G}.dim_date(date_key)    RELY",
)

In [ ]:
# fact_practice_event — one row per practice event; wide pass-through of all silver columns + keys.
# FKs to dim_session, batter dim_player, dim_team, dim_date. (No pitcher id in the payload — batter only.)
upsert(f"{G}.fact_practice_event", f"""
    SELECT
        md5(id)                    AS practice_event_key,
        md5(session_id)            AS session_key,
        md5(batter_id)             AS batter_key,
        md5(session_home_team_id)  AS home_team_key,
        md5(session_away_team_id)  AS away_team_key,
        cast(date_format(coalesce(try_cast(left(session_game_date, 10) AS date), try_to_date(session_game_date, 'M/d/yyyy')), 'yyyyMMdd') AS int) AS date_key,
        *                                            -- all 265 silver columns passed through (id, pitch, contact, runners, ...)
    FROM {S}.silver_practice_events
""", ["practice_event_key"])
print("fact_practice_event:", spark.table(f"{G}.fact_practice_event").count())

constrain(
    f"ALTER TABLE {G}.fact_practice_event ALTER COLUMN practice_event_key SET NOT NULL",
    f"ALTER TABLE {G}.fact_practice_event ADD CONSTRAINT pk_fact_practice_event PRIMARY KEY (practice_event_key) RELY",
    f"ALTER TABLE {G}.fact_practice_event ADD CONSTRAINT fk_pevent_session   FOREIGN KEY (session_key)   REFERENCES {G}.dim_session(session_key) RELY",
    f"ALTER TABLE {G}.fact_practice_event ADD CONSTRAINT fk_pevent_batter    FOREIGN KEY (batter_key)    REFERENCES {G}.dim_player(player_key)   RELY",
    f"ALTER TABLE {G}.fact_practice_event ADD CONSTRAINT fk_pevent_home_team FOREIGN KEY (home_team_key) REFERENCES {G}.dim_team(team_key)       RELY",
    f"ALTER TABLE {G}.fact_practice_event ADD CONSTRAINT fk_pevent_away_team FOREIGN KEY (away_team_key) REFERENCES {G}.dim_team(team_key)       RELY",
    f"ALTER TABLE {G}.fact_practice_event ADD CONSTRAINT fk_pevent_date      FOREIGN KEY (date_key)      REFERENCES {G}.dim_date(date_key)       RELY",
)

In [ ]:
# fact_practice_workout — one row per training-workout group (composite key in silver)
upsert(f"{G}.fact_practice_workout", f"""
    SELECT
        md5(concat_ws('|', session_home_team_id, cast(group_number AS string))) AS workout_key,
        md5(session_id)            AS session_key,
        md5(session_home_team_id)  AS home_team_key,
        md5(session_away_team_id)  AS away_team_key,
        cast(date_format(coalesce(try_cast(left(session_game_date, 10) AS date), try_to_date(session_game_date, 'M/d/yyyy')), 'yyyyMMdd') AS int) AS date_key,
        group_number,
        session_home_team_name, session_away_team_name,
        session_season, session_league_name
    FROM {S}.silver_practice_training_workout
""", ["workout_key"])
print("fact_practice_workout:", spark.table(f"{G}.fact_practice_workout").count())

constrain(
    f"ALTER TABLE {G}.fact_practice_workout ALTER COLUMN workout_key SET NOT NULL",
    f"ALTER TABLE {G}.fact_practice_workout ADD CONSTRAINT pk_fact_practice_workout PRIMARY KEY (workout_key) RELY",
    f"ALTER TABLE {G}.fact_practice_workout ADD CONSTRAINT fk_workout_session   FOREIGN KEY (session_key)   REFERENCES {G}.dim_session(session_key) RELY",
    f"ALTER TABLE {G}.fact_practice_workout ADD CONSTRAINT fk_workout_home_team FOREIGN KEY (home_team_key) REFERENCES {G}.dim_team(team_key)       RELY",
    f"ALTER TABLE {G}.fact_practice_workout ADD CONSTRAINT fk_workout_away_team FOREIGN KEY (away_team_key) REFERENCES {G}.dim_team(team_key)       RELY",
    f"ALTER TABLE {G}.fact_practice_workout ADD CONSTRAINT fk_workout_date      FOREIGN KEY (date_key)      REFERENCES {G}.dim_date(date_key)       RELY",
)

In [ ]:
# fact_roster_stint — player-team stints over time (silver_players_teamhistory).
# The teamhistory payload carries no player id (only the row id and player name), so this links to
# dim_team and dim_date only. There is no reliable FK to dim_player; join to players by name if needed.
upsert(f"{G}.fact_roster_stint", f"""
    SELECT
        md5(id)            AS roster_stint_key,
        id                 AS synergy_teamhistory_id,
        md5(team_id)       AS team_key,
        cast(date_format(coalesce(try_cast(left(start_date, 10) AS date), try_to_date(start_date, 'M/d/yyyy')), 'yyyyMMdd') AS int) AS start_date_key,
        cast(date_format(coalesce(try_cast(left(end_date, 10) AS date), try_to_date(end_date, 'M/d/yyyy')), 'yyyyMMdd') AS int) AS end_date_key,
        concat_ws(' ', name_first, name_last) AS player_name,
        name_first, name_last, birth_date,
        jersey_number, position, bats_handedness, throws_handedness,
        team_name, league_name, active,
        start_date, end_date
    FROM {S}.silver_players_teamhistory
""", ["roster_stint_key"])
print("fact_roster_stint:", spark.table(f"{G}.fact_roster_stint").count())

constrain(
    f"ALTER TABLE {G}.fact_roster_stint ALTER COLUMN roster_stint_key SET NOT NULL",
    f"ALTER TABLE {G}.fact_roster_stint ADD CONSTRAINT pk_fact_roster_stint PRIMARY KEY (roster_stint_key) RELY",
    f"ALTER TABLE {G}.fact_roster_stint ADD CONSTRAINT fk_stint_team       FOREIGN KEY (team_key)       REFERENCES {G}.dim_team(team_key) RELY",
    f"ALTER TABLE {G}.fact_roster_stint ADD CONSTRAINT fk_stint_start_date FOREIGN KEY (start_date_key) REFERENCES {G}.dim_date(date_key) RELY",
    f"ALTER TABLE {G}.fact_roster_stint ADD CONSTRAINT fk_stint_end_date   FOREIGN KEY (end_date_key)   REFERENCES {G}.dim_date(date_key) RELY",
)

In [ ]:
# Sample the finished star: a fact-and-dim join exercises the RELY keys end to end.
spark.sql(f"""
  SELECT g.season,
         t.name AS home_team,
         d.full_date,
         count(*)                              AS pitches,
         round(avg(p.pitch_pitch_speed_mph), 1) AS avg_mph
  FROM {G}.fact_pitch p
  JOIN {G}.fact_game  g ON p.game_key      = g.game_key
  JOIN {G}.dim_team   t ON g.home_team_key = t.team_key
  JOIN {G}.dim_date   d ON p.date_key      = d.date_key
  GROUP BY g.season, t.name, d.full_date
  ORDER BY pitches DESC
  LIMIT 10
""").show(truncate=False)